In [1]:
from ete3 import Tree
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [ ]:

with open('../rerooted_tree_no_germline_node_names.nwk', 'r') as f:

    file_content = f.read()
newick_tree = file_content

# Create a tree object
tree = Tree(newick_tree, format = 1)

# Get the root node
root = tree.get_tree_root()

distances_from_root = {}

# Calculate the distance between the root and each leaf node
for leaf in tree.iter_leaves():
    distances_from_root[str(leaf)[3:41]] = root.get_distance(leaf, topology_only=False) 
signatures_data_frame = pd.read_csv("../example_vcf_per_cell/Assignment_Solution/Activities/Assignment_Solution_Activities.txt", sep="\t")
signatures_data_frame_no_0 = signatures_data_frame.loc[:, (signatures_data_frame != 0).any(axis=0)]
signatures_data_frame_no_0 = signatures_data_frame_no_0.set_index("Samples")

signatures_data_frame_no_0 = df_normalized
sig_values_dict = {}
for lf in tree.iter_leaves():
    leaf_name = str(lf)[3:41]
    if leaf_name in signatures_data_frame_no_0.index:
        percentages = signatures_data_frame_no_0.loc[leaf_name].values
        sig_values_dict[leaf_name] = percentages
        


import pandas as pd

# Get all signature names from the original dataset
all_signatures = list(signatures_data_frame_no_0.columns)

# Initialize an empty DataFrame with all signature names as columns
df = pd.DataFrame(columns=all_signatures)

# Iterate over the tree leaves and extract signature values for all columns
for lf in tree.iter_leaves():
    leaf_name = str(lf)[3:41]  # Extract the leaf identifier
    if leaf_name in sig_values_dict:
        # Extract all signature values for this leaf
        df.loc[leaf_name] = sig_values_dict[leaf_name]
import seaborn as sns
import matplotlib.pyplot as plt

# Define the list of signatures of interest (example, replace with your actual list)
signatures_of_interest = ['SBS1', 'SBS21', 'SBS32', 'SBS12', 'SBS54', 'SBS87', 'SBS37', 'SBS26', 'SBS40c', 'SBS5']  # Replace with your actual signature names

# Filter the DataFrame to include only the signatures of interest
df_interest = df[signatures_of_interest]

In [10]:
import scanpy as sc
from scipy.io import mmread
import pandas as pd
import os

def load_sample(folder, sample_name):
    """Load a single sample and return an AnnData object with sample metadata."""
    matrix = mmread(os.path.join(folder, "filtered_feature_bc_matrix", "matrix.mtx")).T
    genes = pd.read_csv(os.path.join(folder, "filtered_feature_bc_matrix", "features.tsv"), 
                        header=None, sep="\t")[1].values
    barcodes = pd.read_csv(os.path.join(folder, "filtered_feature_bc_matrix", "barcodes.tsv"), 
                           header=None, sep="\t")[0].values
    barcodes = [f"{folder}_{barcode}" for barcode in barcodes]


    adata = sc.AnnData(X=matrix, var=pd.DataFrame(index=genes), obs=pd.DataFrame(index=barcodes))
    adata.obs["sample"] = sample_name  # Add sample metadata
    return adata

# Load all four samples
adata_list = [
    load_sample("neck15", "Normal (N)"),
    load_sample("neck16", "Metastasis (M)"),
    load_sample("neck17", "Margins (R)"),
    load_sample("neck18", "Primary (P)")
]




/Users/saracosta/miniforge3/envs/dgea/lib/python3.12/site-packages/anndata/_core/storage.py:48: FutureWarning: AnnData previously had undefined behavior around matrices of type <class 'scipy.sparse._coo.coo_matrix'>.In 0.12, passing in this type will throw an error. Please convert to a supported type.Continue using for this minor version at your own risk.
  warnings.warn(msg, FutureWarning)
/Users/saracosta/miniforge3/envs/dgea/lib/python3.12/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/saracosta/miniforge3/envs/dgea/lib/python3.12/site-packages/anndata/_core/storage.py:48: FutureWarning: AnnData previously had undefined behavior around matrices of type <class 'scipy.sparse._coo.coo_matrix'>.In 0.12, passing in this type will throw an error. Please convert to a supported type.Continue using for this minor version at your own risk.
  warnings.warn(m

In [11]:
for i, adata in enumerate(adata_list):
    # Convert to CSR to allow indexing
    adata.X = adata.X.tocsr()  
    
    # Identify duplicated genes
    duplicated_genes = adata.var.index.duplicated()
    
    if duplicated_genes.sum() > 0:
        print(f"Sample {i} - Removing {duplicated_genes.sum()} duplicated genes.")
        adata = adata[:, ~duplicated_genes].copy()  # Now slicing will work!
        adata_list[i] = adata  # Update in list



Sample 0 - Removing 22 duplicated genes.
Sample 1 - Removing 22 duplicated genes.
Sample 2 - Removing 22 duplicated genes.
Sample 3 - Removing 22 duplicated genes.


In [12]:
# Concatenate the datasets
adata_combined = sc.concat(adata_list, join="outer", label="sample", keys=["N", "M", "R", "P"])

# Create two groups for low and high value for a certain signature of interest to after perform DGEA

In [22]:
# Choose your signature of interest
signature_name = 'SBS32'  # replace with the name of the column (signature)

# Define thresholds
low_thresh = 5   # everything <= this goes into the "low" group
high_thresh = 6  # everything >= this goes into the "high" group

# Grouping
low_cells = df_interest.index[df_interest[signature_name] <= low_thresh].tolist()
high_cells = df_interest.index[df_interest[signature_name] > high_thresh].tolist()

print(f"# low cells: {len(low_cells)}")
print(f"# high cells: {len(high_cells)}")



# low cells: 30474
# high cells: 25671


In [26]:
low_cells_barcodes = []
for i in low_cells:
    low_cells_barcodes.append(i)

high_cells_barcodes = []
for i in high_cells:
    high_cells_barcodes.append(i)
    

In [ ]:
# Assuming: adata_combined is your AnnData object
# Provide your two groups of cells (lists of obs_names)
group1_cells = low_cells_barcodes  # e.g., ['cellA', 'cellB', ...]
group2_cells = high_cells_barcodes  # e.g., ['cellX', 'cellY', ...]

# Create a new column in obs to label the groups
adata_combined.obs['dge_group'] = 'other'
adata_combined.obs.loc[group1_cells, 'dge_group'] = 'group1'
adata_combined.obs.loc[group2_cells, 'dge_group'] = 'group2'

In [29]:
adata_combined_filt = adata_combined

In [32]:
sc.pp.normalize_total(adata_combined_filt, target_sum=1e4)
sc.pp.log1p(adata_combined_filt)

In [36]:
sc.pp.filter_genes(adata_combined_filt, min_cells=int(0.1 * adata_combined_filt.n_obs))

In [22]:
adata_combined_filt.write_h5ad("adata_sbs32_all_26_sep.h5ad")